# OIST Research Internship Project
## DSP-Based Bearing Fault Diagnosis Using Vibration Signals and Machine Learning — FINAL RESEARCH VERSION

**Research question:** Can DSP-based vibration features enable reliable bearing-condition diagnosis while maintaining a compact computational footprint suitable for resource-constrained edge devices?

**Pipeline:** Raw vibration → DSP preprocessing → segmentation → time/frequency-domain features → recording-aware holdout → grouped model selection → SVM + Random Forest + raw-signal 1D-CNN → ablation → noise robustness → edge-oriented profiling → reproducibility.

### Final methodological safeguards

1. Windows from one recording never cross an evaluation boundary.
2. The held-out test recordings are selected before model fitting.
3. Classical-model hyperparameters are selected using **training recordings only**.
4. Cross-validation used for model-selection diagnostics is restricted to the **training pool**, not the final test set.
5. CNN normalization statistics are calculated from CNN training recordings only.
6. Noise robustness is evaluated only on already-trained models.
7. Edge timings are explicitly described as Colab/software measurements, not MCU benchmarks.
8. Dataset limitations, especially the small number of independent recordings in some classes, are reported rather than hidden.

**Dataset:** Case Western Reserve University (CWRU) bearing vibration dataset, using the same repository as the original project.


## 1. Research objectives

1. Build a reproducible vibration-signal processing pipeline.
2. Apply DSP preprocessing to bearing vibration measurements.
3. Segment long recordings into fixed-length windows.
4. Extract interpretable time- and frequency-domain features.
5. Train classical ML models on DSP features.
6. Train a 1D-CNN directly on vibration windows.
7. Evaluate accuracy, precision, recall, F1-score, balanced accuracy and confusion matrices.
8. Use recording-aware validation to reduce leakage.
9. Quantify the contribution of time- and frequency-domain feature groups.
10. Test robustness to controlled measurement noise.
11. Compare predictive performance with computational cost and memory footprint.

In [ ]:
# 2. Environment setup
# Colab already provides the scientific Python stack and TensorFlow.
# We intentionally do NOT run `pip install -U tensorflow` here because changing
# TensorFlow/protobuf versions can conflict with other preinstalled Colab packages.

import os, json, random, time, warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy.io import loadmat
from scipy.signal import butter, filtfilt, welch
from scipy.stats import kurtosis, skew

from sklearn.model_selection import StratifiedGroupKFold, GroupShuffleSplit, GridSearchCV
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.pipeline import Pipeline
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score,
    precision_recall_fscore_support,
    classification_report, confusion_matrix, ConfusionMatrixDisplay
)

try:
    import tensorflow as tf
except ImportError as exc:
    raise ImportError(
        "TensorFlow is not available in this Colab runtime. "
        "Please switch to a standard Colab Python runtime rather than installing "
        "a new TensorFlow version into an existing environment."
    ) from exc

from tensorflow.keras import Sequential
from tensorflow.keras.layers import (
    Input, Conv1D, MaxPooling1D, GlobalAveragePooling1D,
    Dense, Dropout
)
from tensorflow.keras.callbacks import EarlyStopping

warnings.filterwarnings("ignore")

SEED = 42
np.random.seed(SEED)
random.seed(SEED)
tf.random.set_seed(SEED)

print("Environment ready.")
print("TensorFlow:", tf.__version__)


## 3. Download the CWRU dataset

The original notebook used the `XiongMeijing/CWRU-1` GitHub repository. This version keeps the same source so the experiment remains traceable to the earlier project.

In [ ]:
# 3. Download dataset
DATA_ROOT = Path("cwru_data")
REPO_DIR = DATA_ROOT / "CWRU-1"
REPO_URL = "https://github.com/XiongMeijing/CWRU-1.git"

if not REPO_DIR.exists():
    DATA_ROOT.mkdir(parents=True, exist_ok=True)
    !git clone -q {REPO_URL} {REPO_DIR}
else:
    print("Repository already exists; skipping clone.")

DATA_DIR = REPO_DIR / "Data"
print("Data directory:", DATA_DIR.resolve())
print("Exists:", DATA_DIR.exists())

## 4. Dataset exploration and label audit

Labels are derived from the directory structure rather than invented in the notebook. The primary `condition` is the directory immediately under `Data/`, while the full relative path is retained as the recording identifier.

**Important:** before modeling, we explicitly inspect the number of recordings per class. Stratified group evaluation is only possible when each class has enough independent recordings.

In [ ]:
# 4. Discover recordings
mat_files = sorted(DATA_DIR.rglob("*.mat"))
if not mat_files:
    raise FileNotFoundError(f"No .mat files found under {DATA_DIR.resolve()}")

rows = []
for p in mat_files:
    parts = p.relative_to(DATA_DIR).parts
    condition = parts[0] if len(parts) >= 2 else p.parent.name
    rows.append({
        "file_path": str(p),
        "file_name": p.name,
        "condition": condition,
        "relative_path": str(p.relative_to(DATA_DIR))
    })

recordings = pd.DataFrame(rows)

print("Number of recordings:", len(recordings))
print("\nRecording counts by condition:")
recording_counts = recordings["condition"].value_counts().sort_index()
display(recording_counts.to_frame("recordings"))

if recording_counts.min() < 2:
    raise ValueError(
        "At least 2 independent recordings per class are required for "
        "stratified group evaluation. The discovered dataset does not satisfy this."
    )

print("\nFirst recordings:")
display(recordings.head(20))

In [ ]:
# 4b. Group/label consistency audit
group_label_counts = recordings.groupby("relative_path")["condition"].nunique()
if group_label_counts.max() != 1:
    raise ValueError("A recording identifier maps to more than one class.")

print("Each recording has exactly one condition label.")
print("Minimum recordings in any class:", int(recording_counts.min()))
print("Maximum recordings in any class:", int(recording_counts.max()))

In [ ]:
# Inspect one .mat file
example_path = Path(recordings.iloc[0]["file_path"])
example_mat = loadmat(example_path)

mat_keys = [k for k in example_mat.keys() if not k.startswith("__")]
signal_candidates = [
    k for k in mat_keys
    if k.endswith("_DE_time") or k.endswith("_FE_time")
]

print("Example:", example_path)
print("MAT keys:", mat_keys)
print("Vibration keys:", signal_candidates)

if not signal_candidates:
    raise ValueError("No DE/FE vibration signal found.")

preferred_key = next((k for k in signal_candidates if "_DE_time" in k), signal_candidates[0])
example_signal = np.asarray(example_mat[preferred_key]).squeeze().astype(np.float64)

print("Selected signal:", preferred_key)
print("Signal length:", len(example_signal))
print("First 10 samples:", example_signal[:10])

## 5. Signal visualization

The original project used a 12 kHz sampling frequency. We retain that setting and inspect the raw signal in both time and frequency domains.

In [ ]:
# 5. Visualize raw signal
FS = 12000
VIS_SECONDS = 0.25
VIS_N = min(len(example_signal), int(FS * VIS_SECONDS))
t = np.arange(VIS_N) / FS

plt.figure(figsize=(12, 4))
plt.plot(t, example_signal[:VIS_N])
plt.title("Raw vibration signal")
plt.xlabel("Time (s)")
plt.ylabel("Amplitude")
plt.grid(True, alpha=0.3)
plt.show()

freqs = np.fft.rfftfreq(VIS_N, d=1/FS)
spectrum = np.abs(np.fft.rfft(example_signal[:VIS_N]))

plt.figure(figsize=(12, 4))
plt.plot(freqs, spectrum)
plt.title("Raw vibration spectrum")
plt.xlabel("Frequency (Hz)")
plt.ylabel("Magnitude")
plt.xlim(0, FS/2)
plt.grid(True, alpha=0.3)
plt.show()

## 6. DSP preprocessing

A fourth-order Butterworth low-pass filter with a 2 kHz cutoff is retained from the original notebook. `filtfilt` is used so the filtering does not introduce a phase shift.

In [ ]:
# 6. Butterworth low-pass filter
CUTOFF_FREQ = 2000
FILTER_ORDER = 4

def butter_lowpass_filter(signal, fs=FS, cutoff=CUTOFF_FREQ, order=FILTER_ORDER):
    nyquist = 0.5 * fs
    if cutoff >= nyquist:
        raise ValueError("Cutoff must be below Nyquist frequency.")
    b, a = butter(order, cutoff / nyquist, btype="low")
    return filtfilt(b, a, signal)

filtered_example = butter_lowpass_filter(example_signal)

plt.figure(figsize=(12, 4))
plt.plot(t, example_signal[:VIS_N], label="Raw")
plt.plot(t, filtered_example[:VIS_N], label="Filtered")
plt.title("Raw vs. DSP-filtered signal")
plt.xlabel("Time (s)")
plt.ylabel("Amplitude")
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

## 7. Data segmentation

Each recording is divided into:

- **1 second windows**
- **12,000 samples per window**
- **50% overlap**

The recording identifier is preserved for every window so that evaluation can be performed at recording level.

In [ ]:
# 7. Segmentation
WINDOW_SECONDS = 1.0
WINDOW_SIZE = int(FS * WINDOW_SECONDS)
OVERLAP = 0.50
STEP_SIZE = int(WINDOW_SIZE * (1 - OVERLAP))

# Runtime control. Increase after confirming available Colab resources.
MAX_WINDOWS_PER_FILE = 100

print("Window size:", WINDOW_SIZE)
print("Step size:", STEP_SIZE)
print("Overlap:", OVERLAP)

In [ ]:
# Signal loading and segmentation helpers
def load_vibration_signal(file_path):
    data = loadmat(file_path)
    keys = [k for k in data.keys() if not k.startswith("__")]
    de_keys = [k for k in keys if k.endswith("_DE_time")]
    fe_keys = [k for k in keys if k.endswith("_FE_time")]

    if de_keys:
        key = de_keys[0]
    elif fe_keys:
        key = fe_keys[0]
    else:
        raise ValueError(f"No DE/FE time-series found in {file_path}")

    signal = np.asarray(data[key]).squeeze().astype(np.float64)
    if signal.ndim != 1 or len(signal) < WINDOW_SIZE:
        raise ValueError(f"Invalid or too-short signal in {file_path}")

    return signal, key

def segment_signal(signal, window_size=WINDOW_SIZE, step_size=STEP_SIZE,
                   max_windows=MAX_WINDOWS_PER_FILE):
    windows = []
    for start in range(0, len(signal) - window_size + 1, step_size):
        if len(windows) >= max_windows:
            break
        windows.append(signal[start:start + window_size])
    return np.asarray(windows)

## 8. Feature extraction

### Time-domain features
Mean, RMS, standard deviation, peak, peak-to-peak, crest factor, kurtosis and skewness.

### Frequency-domain features
Spectral centroid, dominant frequency, spectral RMS and spectral bandwidth.

The aim is to obtain a compact and interpretable representation for classical ML.

In [ ]:
# 8. DSP feature extraction
TIME_FEATURES = [
    "mean", "rms", "std", "peak", "peak_to_peak",
    "crest_factor", "kurtosis", "skewness"
]

FREQ_FEATURES = [
    "spectral_centroid", "dominant_frequency",
    "spectral_rms", "spectral_bandwidth"
]

FEATURE_COLUMNS = TIME_FEATURES + FREQ_FEATURES

def extract_features(window, fs=FS):
    x = np.asarray(window, dtype=np.float64)

    mean_val = np.mean(x)
    rms_val = np.sqrt(np.mean(x**2))
    std_val = np.std(x)
    peak_val = np.max(np.abs(x))
    p2p_val = np.ptp(x)

    crest = peak_val / (rms_val + 1e-12)
    kurt = kurtosis(x, fisher=True, bias=False)
    skewness = skew(x, bias=False)

    f, pxx = welch(x, fs=fs, nperseg=min(2048, len(x)))
    power = np.maximum(pxx, 0)
    total_power = np.sum(power) + 1e-12

    spectral_centroid = np.sum(f * power) / total_power
    dominant_frequency = f[np.argmax(power)]
    spectral_rms = np.sqrt(np.sum(power) / len(power))
    spectral_bandwidth = np.sqrt(
        np.sum(((f - spectral_centroid) ** 2) * power) / total_power
    )

    return {
        "mean": mean_val,
        "rms": rms_val,
        "std": std_val,
        "peak": peak_val,
        "peak_to_peak": p2p_val,
        "crest_factor": crest,
        "kurtosis": kurt,
        "skewness": skewness,
        "spectral_centroid": spectral_centroid,
        "dominant_frequency": dominant_frequency,
        "spectral_rms": spectral_rms,
        "spectral_bandwidth": spectral_bandwidth
    }

In [ ]:
# 9. Build feature dataset and raw-window dataset
feature_rows = []
raw_windows = []
raw_labels = []
raw_groups = []
failed_files = []

for _, row in recordings.iterrows():
    try:
        signal, signal_key = load_vibration_signal(row["file_path"])
        signal = butter_lowpass_filter(signal)
        windows = segment_signal(signal)

        for window_idx, window in enumerate(windows):
            feats = extract_features(window)
            feats["label"] = row["condition"]
            feats["recording_id"] = row["relative_path"]
            feats["window_id"] = window_idx
            feature_rows.append(feats)

            raw_windows.append(window.astype(np.float32))
            raw_labels.append(row["condition"])
            raw_groups.append(row["relative_path"])

    except Exception as exc:
        failed_files.append((row["relative_path"], str(exc)))

features_df = pd.DataFrame(feature_rows)

print("Feature dataset shape:", features_df.shape)
print("Recordings represented:", features_df["recording_id"].nunique())
print("\nWindow counts by condition:")
display(features_df["label"].value_counts().sort_index().to_frame("windows"))

if failed_files:
    print("Skipped files:", len(failed_files))
    display(pd.DataFrame(failed_files, columns=["file", "error"]).head(20))
else:
    print("No files were skipped.")

## 10. Feature analysis

Inspect feature statistics, correlations and RMS distributions before modeling.

In [ ]:
# 10. Feature analysis
display(features_df[FEATURE_COLUMNS].describe().T)

corr = features_df[FEATURE_COLUMNS].corr()
plt.figure(figsize=(11, 8))
plt.imshow(corr, aspect="auto")
plt.colorbar(label="Correlation")
plt.xticks(range(len(FEATURE_COLUMNS)), FEATURE_COLUMNS, rotation=90)
plt.yticks(range(len(FEATURE_COLUMNS)), FEATURE_COLUMNS)
plt.title("DSP feature correlation matrix")
plt.tight_layout()
plt.show()

plt.figure(figsize=(10, 5))
for label, group in features_df.groupby("label"):
    plt.hist(group["rms"], bins=30, alpha=0.45, label=str(label))
plt.title("RMS distribution by condition")
plt.xlabel("RMS")
plt.ylabel("Window count")
plt.legend()
plt.grid(True, alpha=0.25)
plt.show()

## 11. Stratified group-aware evaluation design

A plain `GroupShuffleSplit` prevents recording leakage, but it does not guarantee that every class appears in the test set.

Here we use `StratifiedGroupKFold` to construct a **group-aware, class-aware holdout**. The chosen fold must contain every class in both train and test. If the dataset does not contain enough independent recordings to satisfy that requirement, the notebook stops rather than silently reporting incomplete test results.

In [ ]:
# 11. Build a stratified group-aware holdout
X_all = features_df[FEATURE_COLUMNS].copy()
y_all_text = features_df["label"].astype(str)
groups_all = features_df["recording_id"].astype(str)

group_label_table = features_df[["recording_id", "label"]].drop_duplicates()
group_counts = group_label_table["label"].value_counts()

N_SPLITS = min(5, int(group_counts.min()))
if N_SPLITS < 2:
    raise ValueError("Not enough independent recordings per class for stratified group evaluation.")

sgkf = StratifiedGroupKFold(
    n_splits=N_SPLITS,
    shuffle=True,
    random_state=SEED
)

candidate_folds = []
overall_props = y_all_text.value_counts(normalize=True).sort_index()

for fold_id, (tr_idx, te_idx) in enumerate(
    sgkf.split(X_all, y_all_text, groups_all)
):
    tr_classes = set(y_all_text.iloc[tr_idx])
    te_classes = set(y_all_text.iloc[te_idx])
    all_classes = set(y_all_text.unique())

    if tr_classes != all_classes or te_classes != all_classes:
        continue

    test_props = y_all_text.iloc[te_idx].value_counts(normalize=True).reindex(
        overall_props.index, fill_value=0
    )
    deviation = float(np.abs(test_props - overall_props).sum())

    candidate_folds.append({
        "fold": fold_id,
        "train_idx": tr_idx,
        "test_idx": te_idx,
        "deviation": deviation
    })

if not candidate_folds:
    raise ValueError(
        "No StratifiedGroupKFold split contains every class in both train and test. "
        "More independent recordings per class are required for a complete multiclass evaluation."
    )

best_candidate = min(candidate_folds, key=lambda d: d["deviation"])
train_idx = best_candidate["train_idx"]
test_idx = best_candidate["test_idx"]

X_train = X_all.iloc[train_idx].copy()
X_test = X_all.iloc[test_idx].copy()
y_train_text = y_all_text.iloc[train_idx].copy()
y_test_text = y_all_text.iloc[test_idx].copy()
groups_train = groups_all.iloc[train_idx].copy()
groups_test = groups_all.iloc[test_idx].copy()

print("Selected holdout fold:", best_candidate["fold"])
print("Training recordings:", groups_train.nunique())
print("Testing recordings :", groups_test.nunique())
print("Training windows   :", len(train_idx))
print("Testing windows    :", len(test_idx))
print("\nTraining class distribution:")
display(y_train_text.value_counts().sort_index().to_frame("windows"))
print("Testing class distribution:")
display(y_test_text.value_counts().sort_index().to_frame("windows"))

overlap = set(groups_train) & set(groups_test)
print("Recording overlap:", overlap)
if overlap:
    raise AssertionError("Data leakage detected.")

In [ ]:
# 12. Encode labels
label_encoder = LabelEncoder()
label_encoder.fit(y_all_text)

y_train = label_encoder.transform(y_train_text)
y_test = label_encoder.transform(y_test_text)

class_names = list(label_encoder.classes_)
NUM_CLASSES = len(class_names)

print("Classes:", class_names)
print("All classes present in test:", set(np.unique(y_test)) == set(range(NUM_CLASSES)))

## 13. Shared evaluation utilities

In addition to accuracy and weighted F1, **macro-F1 and balanced accuracy** are reported because class imbalance can make weighted metrics look better than performance on smaller classes.

In [ ]:
def metric_dict(y_true, y_pred):
    precision_w, recall_w, f1_w, _ = precision_recall_fscore_support(
        y_true, y_pred, average="weighted", zero_division=0
    )
    precision_m, recall_m, f1_m, _ = precision_recall_fscore_support(
        y_true, y_pred, average="macro", zero_division=0
    )
    return {
        "accuracy": accuracy_score(y_true, y_pred),
        "balanced_accuracy": balanced_accuracy_score(y_true, y_pred),
        "precision_weighted": precision_w,
        "recall_weighted": recall_w,
        "f1_weighted": f1_w,
        "f1_macro": f1_m
    }

def evaluate_classifier(name, y_true, y_pred, labels, target_names, show_matrix=True):
    metrics = metric_dict(y_true, y_pred)
    print(name)
    for k, v in metrics.items():
        print(f"{k:20s}: {v:.4f}")
    print("\nClassification report:")
    print(classification_report(
        y_true, y_pred, labels=labels,
        target_names=target_names, zero_division=0
    ))

    if show_matrix:
        cm = confusion_matrix(y_true, y_pred, labels=labels)
        ConfusionMatrixDisplay(
            confusion_matrix=cm,
            display_labels=target_names
        ).plot(values_format="d")
        plt.title(f"{name} confusion matrix")
        plt.show()

    out = {"model": name}
    out.update(metrics)
    return out

## 14. Classical ML — SVM baseline

An RBF-kernel SVM is trained on standardized DSP features.

In [ ]:
# 14. SVM baseline
svm_model = Pipeline([
    ("scaler", StandardScaler()),
    ("classifier", SVC(kernel="rbf", C=10, gamma="scale"))
])

start = time.perf_counter()
svm_model.fit(X_train, y_train)
svm_train_time = time.perf_counter() - start
svm_pred = svm_model.predict(X_test)

print("SVM training time: %.4f s" % svm_train_time)

## 15. Classical ML — Random Forest baseline

In [ ]:
# 15. Random Forest baseline
rf_model = RandomForestClassifier(
    n_estimators=200,
    random_state=SEED,
    n_jobs=-1,
    class_weight="balanced"
)

start = time.perf_counter()
rf_model.fit(X_train, y_train)
rf_train_time = time.perf_counter() - start
rf_pred = rf_model.predict(X_test)

print("Random Forest training time: %.4f s" % rf_train_time)

In [ ]:
# Baseline holdout results
baseline_results = []
baseline_results.append(
    evaluate_classifier("SVM", y_test, svm_pred, np.arange(NUM_CLASSES), class_names)
)
baseline_results.append(
    evaluate_classifier("Random Forest", y_test, rf_pred, np.arange(NUM_CLASSES), class_names)
)
baseline_results_df = pd.DataFrame(baseline_results)
display(baseline_results_df)

## 16. Grouped cross-validation for classical ML

The final test recordings remain untouched during model-selection diagnostics.

SVM and Random Forest are evaluated with **StratifiedGroupKFold using only the training recordings**. This gives a more honest estimate of sensitivity to the training-recording split without allowing final-test windows into model-selection analysis.


In [ ]:
# 16. Stratified group cross-validation on TRAINING RECORDINGS ONLY
cv_splits = list(
    StratifiedGroupKFold(
        n_splits=N_SPLITS,
        shuffle=True,
        random_state=SEED
    ).split(X_train, y_train_text, groups_train)
)

cv_rows = []

for fold_id, (tr, va) in enumerate(cv_splits, start=1):
    Xtr, Xva = X_train.iloc[tr], X_train.iloc[va]
    ytr = y_train[tr]
    yva = y_train[va]
    gtr = groups_train.iloc[tr].to_numpy()
    gva = groups_train.iloc[va].to_numpy()

    assert set(gtr).isdisjoint(set(gva))

    svm_cv = Pipeline([
        ("scaler", StandardScaler()),
        ("classifier", SVC(kernel="rbf", C=10, gamma="scale"))
    ])

    rf_cv = RandomForestClassifier(
        n_estimators=200,
        random_state=SEED,
        n_jobs=-1,
        class_weight="balanced"
    )

    svm_cv.fit(Xtr, ytr)
    rf_cv.fit(Xtr, ytr)

    for model_name, pred in [
        ("SVM", svm_cv.predict(Xva)),
        ("Random Forest", rf_cv.predict(Xva))
    ]:
        m = metric_dict(yva, pred)
        cv_rows.append({
            "fold": fold_id,
            "model": model_name,
            **m
        })

cv_results_df = pd.DataFrame(cv_rows)
display(cv_results_df)

cv_summary = (
    cv_results_df
    .groupby("model")[["accuracy", "balanced_accuracy", "f1_weighted", "f1_macro"]]
    .agg(["mean", "std"])
)
display(cv_summary)

print("\nImportant: final held-out test recordings were NOT used in this CV.")


## 17. Modest group-aware hyperparameter tuning

Hyperparameters are selected using only training data and recording-aware folds. The final holdout test set remains untouched during tuning.

In [ ]:
# 17. Hyperparameter tuning
train_group_labels = (
    features_df.iloc[train_idx][["recording_id", "label"]]
    .drop_duplicates()
)
train_group_counts = train_group_labels["label"].value_counts()
inner_splits = min(3, int(train_group_counts.min()))

if inner_splits >= 2:
    inner_cv = StratifiedGroupKFold(
        n_splits=inner_splits,
        shuffle=True,
        random_state=SEED
    )

    svm_search = GridSearchCV(
        Pipeline([
            ("scaler", StandardScaler()),
            ("classifier", SVC(kernel="rbf"))
        ]),
        param_grid={
            "classifier__C": [1, 10, 100],
            "classifier__gamma": ["scale", 0.01, 0.1]
        },
        scoring="f1_weighted",
        cv=inner_cv,
        n_jobs=-1
    )

    rf_search = GridSearchCV(
        RandomForestClassifier(
            random_state=SEED,
            n_jobs=-1,
            class_weight="balanced"
        ),
        param_grid={
            "n_estimators": [100, 200],
            "max_depth": [None, 10],
            "max_features": ["sqrt"]
        },
        scoring="f1_weighted",
        cv=inner_cv,
        n_jobs=-1
    )

    svm_search.fit(X_train, y_train, groups=groups_train)
    rf_search.fit(X_train, y_train, groups=groups_train)

    print("Best SVM parameters:", svm_search.best_params_)
    print("Best SVM inner-CV F1:", svm_search.best_score_)
    print("Best RF parameters:", rf_search.best_params_)
    print("Best RF inner-CV F1:", rf_search.best_score_)

    tuned_svm = svm_search.best_estimator_
    tuned_rf = rf_search.best_estimator_

    tuned_svm_pred = tuned_svm.predict(X_test)
    tuned_rf_pred = tuned_rf.predict(X_test)

    tuned_results = [
        evaluate_classifier("Tuned SVM", y_test, tuned_svm_pred,
                             np.arange(NUM_CLASSES), class_names),
        evaluate_classifier("Tuned Random Forest", y_test, tuned_rf_pred,
                             np.arange(NUM_CLASSES), class_names)
    ]
    tuned_results_df = pd.DataFrame(tuned_results)
    display(tuned_results_df)
else:
    print(
        "Training set does not contain enough independent recordings per class "
        "for inner grouped tuning. Using baseline hyperparameters."
    )
    tuned_svm = svm_model
    tuned_rf = rf_model
    tuned_svm_pred = svm_pred
    tuned_rf_pred = rf_pred
    tuned_results_df = baseline_results_df.copy()

## 18. Random Forest feature importance

Feature importance is used descriptively to inspect which DSP variables contribute most to the trained forest.

In [ ]:
# 18. Random Forest feature importance
importance_df = pd.DataFrame({
    "feature": FEATURE_COLUMNS,
    "importance": tuned_rf.feature_importances_
}).sort_values("importance", ascending=False)

display(importance_df)

plt.figure(figsize=(10, 5))
plt.bar(importance_df["feature"], importance_df["importance"])
plt.xticks(rotation=70)
plt.ylabel("Importance")
plt.title("Random Forest feature importance")
plt.grid(True, axis="y", alpha=0.25)
plt.tight_layout()
plt.show()

## 19. Raw-signal 1D-CNN with recording-aware validation

The CNN receives the waveform itself rather than the handcrafted feature vector.

**Important change:** the validation set is separated by recording ID. The earlier notebook used `validation_split=0.20`, which could place windows from the same recording in both training and validation.

In [ ]:
# 19. Prepare CNN arrays — robust recording-aware validation
raw_X = np.asarray(raw_windows, dtype=np.float32)
raw_y_text = np.asarray(raw_labels).astype(str)
raw_groups = np.asarray(raw_groups).astype(str)

cnn_label_encoder = LabelEncoder()
cnn_label_encoder.fit(raw_y_text)
raw_y = cnn_label_encoder.transform(raw_y_text)

cnn_class_names = list(cnn_label_encoder.classes_)
CNN_NUM_CLASSES = len(cnn_class_names)
all_cnn_classes = set(range(CNN_NUM_CLASSES))

# Use the same selected holdout recording groups as the classical models.
test_group_set = set(groups_test)
cnn_test_mask = np.array([g in test_group_set for g in raw_groups])
cnn_train_pool_mask = ~cnn_test_mask

pool_idx = np.where(cnn_train_pool_mask)[0]
test_idx_cnn = np.where(cnn_test_mask)[0]

pool_groups = raw_groups[pool_idx]
pool_labels = raw_y[pool_idx]

# Count independent recordings per class in the CNN training pool.
pool_group_label_df = pd.DataFrame({
    "group": pool_groups,
    "label": pool_labels
}).drop_duplicates()

# Each recording must have exactly one label.
group_label_counts = pool_group_label_df.groupby("group")["label"].nunique()
if group_label_counts.max() != 1:
    raise ValueError("A CNN recording identifier maps to more than one class.")

group_counts_by_class = pool_group_label_df["label"].value_counts().sort_index()
print("Independent training recordings per class:")
print(
    pd.DataFrame({
        "class": [cnn_class_names[i] for i in group_counts_by_class.index],
        "recordings": group_counts_by_class.values
    }).to_string(index=False)
)

# We first try StratifiedGroupKFold. Instead of requiring every class to appear
# in validation (which can be impossible with few recordings), we select the
# best feasible fold that keeps every class in TRAIN and maximizes validation
# class coverage.
max_splits = min(5, int(group_counts_by_class.min()))
candidate_splits = []

if max_splits >= 2:
    sgkf_val = StratifiedGroupKFold(
        n_splits=max_splits,
        shuffle=True,
        random_state=SEED
    )
    for fold_id, (tr_rel, va_rel) in enumerate(
        sgkf_val.split(pool_idx, pool_labels, pool_groups), start=1
    ):
        tr_abs = pool_idx[tr_rel]
        va_abs = pool_idx[va_rel]
        tr_classes = set(raw_y[tr_abs])
        va_classes = set(raw_y[va_abs])

        # Training must contain every class; validation may be incomplete if
        # the recording structure makes a complete validation set impossible.
        if tr_classes != all_cnn_classes:
            continue

        val_coverage = len(va_classes & all_cnn_classes) / len(all_cnn_classes)
        val_props = pd.Series(raw_y[va_abs]).value_counts(normalize=True).reindex(
            range(CNN_NUM_CLASSES), fill_value=0
        )
        pool_props = pd.Series(pool_labels).value_counts(normalize=True).reindex(
            range(CNN_NUM_CLASSES), fill_value=0
        )
        deviation = float(np.abs(val_props - pool_props).sum())

        candidate_splits.append({
            "method": "StratifiedGroupKFold",
            "fold": fold_id,
            "train_idx": tr_abs,
            "val_idx": va_abs,
            "coverage": val_coverage,
            "deviation": deviation
        })

# Fallback: repeated GroupShuffleSplit. This remains fully recording-aware
# and is used only when the stratified folds cannot provide a usable split.
if not candidate_splits:
    gss = GroupShuffleSplit(
        n_splits=100,
        test_size=0.20,
        random_state=SEED
    )

    for split_id, (tr_rel, va_rel) in enumerate(
        gss.split(pool_idx, pool_labels, pool_groups), start=1
    ):
        tr_abs = pool_idx[tr_rel]
        va_abs = pool_idx[va_rel]

        tr_classes = set(raw_y[tr_abs])
        va_classes = set(raw_y[va_abs])

        if tr_classes != all_cnn_classes:
            continue

        val_coverage = len(va_classes & all_cnn_classes) / len(all_cnn_classes)
        val_props = pd.Series(raw_y[va_abs]).value_counts(normalize=True).reindex(
            range(CNN_NUM_CLASSES), fill_value=0
        )
        pool_props = pd.Series(pool_labels).value_counts(normalize=True).reindex(
            range(CNN_NUM_CLASSES), fill_value=0
        )
        deviation = float(np.abs(val_props - pool_props).sum())

        candidate_splits.append({
            "method": "GroupShuffleSplit fallback",
            "fold": split_id,
            "train_idx": tr_abs,
            "val_idx": va_abs,
            "coverage": val_coverage,
            "deviation": deviation
        })

if not candidate_splits:
    raise ValueError(
        "Could not create a recording-aware CNN train/validation split while "
        "keeping every class in the training set."
    )

# Prefer maximum validation class coverage, then the closest class distribution.
best = sorted(
    candidate_splits,
    key=lambda d: (-d["coverage"], d["deviation"])
)[0]

cnn_train_idx = best["train_idx"]
cnn_val_idx = best["val_idx"]

X_cnn_train = raw_X[cnn_train_idx]
X_cnn_val = raw_X[cnn_val_idx]
X_cnn_test = raw_X[test_idx_cnn]

y_cnn_train = raw_y[cnn_train_idx]
y_cnn_val = raw_y[cnn_val_idx]
y_cnn_test = raw_y[test_idx_cnn]

cnn_mean = X_cnn_train.mean()
cnn_std = X_cnn_train.std() + 1e-8

X_cnn_train = ((X_cnn_train - cnn_mean) / cnn_std)[..., np.newaxis]
X_cnn_val = ((X_cnn_val - cnn_mean) / cnn_std)[..., np.newaxis]
X_cnn_test = ((X_cnn_test - cnn_mean) / cnn_std)[..., np.newaxis]

print("\nSelected CNN validation split:")
print("Method:", best["method"])
print("Split/fold:", best["fold"])
print("Validation class coverage: %.1f%%" % (100 * best["coverage"]))
if best["coverage"] < 1.0:
    print(
        "WARNING: At least one class is absent from the validation recordings. "
        "This is a dataset-structure limitation, not a leakage issue."
    )

print("\nCNN train shape:", X_cnn_train.shape)
print("CNN validation shape:", X_cnn_val.shape)
print("CNN test shape:", X_cnn_test.shape)
print("CNN classes:", cnn_class_names)

print("\nCNN train class distribution:")
display(
    pd.Series(y_cnn_train).map(dict(enumerate(cnn_class_names)))
    .value_counts().sort_index().to_frame("windows")
)
print("CNN validation class distribution:")
display(
    pd.Series(y_cnn_val).map(dict(enumerate(cnn_class_names)))
    .value_counts().sort_index().to_frame("windows")
)
print("CNN test class distribution:")
display(
    pd.Series(y_cnn_test).map(dict(enumerate(cnn_class_names)))
    .value_counts().sort_index().to_frame("windows")
)

print("Train recordings:", len(set(raw_groups[cnn_train_idx])))
print("Validation recordings:", len(set(raw_groups[cnn_val_idx])))
print("Test recordings:", len(set(raw_groups[test_idx_cnn])))

# Hard leakage checks.
assert not (set(raw_groups[cnn_train_idx]) & set(raw_groups[cnn_val_idx]))
assert not (set(raw_groups[cnn_train_idx]) & set(raw_groups[test_idx_cnn]))
assert not (set(raw_groups[cnn_val_idx]) & set(raw_groups[test_idx_cnn]))

# Final test must remain complete because it is the primary held-out estimate.
if set(y_cnn_test) != all_cnn_classes:
    raise ValueError(
        "The selected CNN test set is missing one or more classes. "
        "This indicates the holdout construction should be revisited."
    )


In [ ]:
# 20. Build CNN
def build_cnn(num_classes):
    model = Sequential([
        Input(shape=(WINDOW_SIZE, 1)),
        Conv1D(16, kernel_size=15, activation="relu"),
        MaxPooling1D(pool_size=4),
        Conv1D(32, kernel_size=9, activation="relu"),
        MaxPooling1D(pool_size=4),
        GlobalAveragePooling1D(),
        Dense(32, activation="relu"),
        Dropout(0.30),
        Dense(num_classes, activation="softmax")
    ])
    model.compile(
        optimizer="adam",
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"]
    )
    return model

cnn_model = build_cnn(CNN_NUM_CLASSES)
cnn_model.summary()

In [ ]:
# 20b. Train CNN with explicit group-separated validation
early_stopping = EarlyStopping(
    monitor="val_loss",
    patience=5,
    restore_best_weights=True
)

start = time.perf_counter()
history = cnn_model.fit(
    X_cnn_train, y_cnn_train,
    validation_data=(X_cnn_val, y_cnn_val),
    epochs=30,
    batch_size=32,
    callbacks=[early_stopping],
    verbose=1
)
cnn_train_time = time.perf_counter() - start

cnn_prob = cnn_model.predict(X_cnn_test, verbose=0)
cnn_pred = np.argmax(cnn_prob, axis=1)

print("CNN training time: %.4f s" % cnn_train_time)
cnn_result = evaluate_classifier(
    "1D-CNN", y_cnn_test, cnn_pred,
    np.arange(CNN_NUM_CLASSES), cnn_class_names
)

In [ ]:
# 21. CNN learning curves
plt.figure(figsize=(10, 4))
plt.plot(history.history["accuracy"], label="Training accuracy")
plt.plot(history.history["val_accuracy"], label="Group-separated validation accuracy")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.title("1D-CNN learning curves")
plt.legend()
plt.grid(True, alpha=0.25)
plt.show()

plt.figure(figsize=(10, 4))
plt.plot(history.history["loss"], label="Training loss")
plt.plot(history.history["val_loss"], label="Group-separated validation loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("1D-CNN loss curves")
plt.legend()
plt.grid(True, alpha=0.25)
plt.show()

## 22. Model comparison

The comparison is descriptive, not a ranking. The purpose is to examine predictive performance, generalization and computational requirements under the same recording-aware test set.

In [ ]:
# 22. Final clean-test comparison
comparison_results = [
    evaluate_classifier("Tuned SVM", y_test, tuned_svm_pred,
                         np.arange(NUM_CLASSES), class_names, show_matrix=False),
    evaluate_classifier("Tuned Random Forest", y_test, tuned_rf_pred,
                         np.arange(NUM_CLASSES), class_names, show_matrix=False),
    cnn_result
]

results_df = pd.DataFrame(comparison_results)
display(results_df)

## 23. DSP feature ablation study

We compare:

- **Time-domain only:** 8 features
- **Frequency-domain only:** 4 features
- **Combined DSP:** 12 features

The same recording-aware holdout is used. This isolates the contribution of the two DSP feature groups.

In [ ]:
# 23. Feature ablation
ablation_rows = []

ablation_sets = {
    "Time-domain only": TIME_FEATURES,
    "Frequency-domain only": FREQ_FEATURES,
    "Combined DSP": FEATURE_COLUMNS
}

for feature_set_name, cols in ablation_sets.items():
    Xtr = X_train[cols]
    Xte = X_test[cols]

    ab_svm = Pipeline([
        ("scaler", StandardScaler()),
        ("classifier", SVC(kernel="rbf", C=10, gamma="scale"))
    ])
    ab_rf = RandomForestClassifier(
        n_estimators=200,
        random_state=SEED,
        n_jobs=-1,
        class_weight="balanced"
    )

    ab_svm.fit(Xtr, y_train)
    ab_rf.fit(Xtr, y_train)

    for model_name, pred in [
        ("SVM", ab_svm.predict(Xte)),
        ("Random Forest", ab_rf.predict(Xte))
    ]:
        m = metric_dict(y_test, pred)
        ablation_rows.append({
            "feature_set": feature_set_name,
            "model": model_name,
            **m
        })

ablation_df = pd.DataFrame(ablation_rows)
display(ablation_df)

## 24. Robustness experiment: controlled additive noise

A simple robustness test is performed by adding zero-mean Gaussian noise to the **held-out test windows only**.

The models are not retrained on noisy data. This asks whether the learned decision rule remains stable when the measured vibration signal is degraded.

Results are reported at 20 dB and 10 dB signal-to-noise ratio (SNR).

In [ ]:
# 24. Noise injection helpers
def add_awgn(x, snr_db, rng):
    x = np.asarray(x, dtype=np.float32)
    signal_power = np.mean(x**2) + 1e-12
    noise_power = signal_power / (10 ** (snr_db / 10))
    noise = rng.normal(0, np.sqrt(noise_power), size=x.shape).astype(np.float32)
    return x + noise

def features_from_windows(windows):
    rows = []
    for w in windows:
        rows.append(extract_features(butter_lowpass_filter(w)))
    return pd.DataFrame(rows)[FEATURE_COLUMNS]

# Test windows in the same order as the CNN test set.
clean_test_windows = raw_X[test_idx_cnn]
clean_test_labels = raw_y[test_idx_cnn]

# Verify label alignment with the classical test set through recording IDs.
cnn_test_group_order = raw_groups[test_idx_cnn]
classical_test_groups_ordered = groups_test.to_numpy()

print("Number of CNN test windows:", len(clean_test_windows))
print("Unique CNN test recordings:", len(np.unique(cnn_test_group_order)))

In [ ]:
# 24b. Evaluate clean and noisy test conditions
robustness_rows = []
rng = np.random.default_rng(SEED)

# Clean feature representation
clean_feature_test = features_from_windows(clean_test_windows)
clean_svm_pred = tuned_svm.predict(clean_feature_test)
clean_rf_pred = tuned_rf.predict(clean_feature_test)

for condition_name, snr_db in [("Clean", None), ("20 dB SNR", 20), ("10 dB SNR", 10)]:
    if snr_db is None:
        noisy_windows = clean_test_windows.copy()
    else:
        noisy_windows = np.asarray(
            [add_awgn(w, snr_db, rng) for w in clean_test_windows],
            dtype=np.float32
        )

    noisy_features = features_from_windows(noisy_windows)
    svm_pred_r = tuned_svm.predict(noisy_features)
    rf_pred_r = tuned_rf.predict(noisy_features)

    noisy_norm = ((noisy_windows - cnn_mean) / cnn_std)[..., np.newaxis]
    cnn_pred_r = np.argmax(cnn_model.predict(noisy_norm, verbose=0), axis=1)

    for model_name, pred in [
        ("Tuned SVM", svm_pred_r),
        ("Tuned Random Forest", rf_pred_r),
        ("1D-CNN", cnn_pred_r)
    ]:
        m = metric_dict(clean_test_labels, pred)
        robustness_rows.append({
            "condition": condition_name,
            "model": model_name,
            **m
        })

robustness_df = pd.DataFrame(robustness_rows)
display(robustness_df)

In [ ]:
# 24c. Robustness visualization
plt.figure(figsize=(9, 5))
for model_name, g in robustness_df.groupby("model"):
    plt.plot(g["condition"], g["f1_weighted"], marker="o", label=model_name)
plt.ylabel("Weighted F1")
plt.xlabel("Test condition")
plt.title("Robustness to additive Gaussian noise")
plt.legend()
plt.grid(True, alpha=0.25)
plt.tight_layout()
plt.show()

## 25. DSP representation memory comparison

The classical models use **12 engineered values per window**, whereas the CNN receives the full **12,000-sample waveform**.

This distinction matters for edge computing because input size directly affects memory transfer and storage.

In [ ]:
# 25. Representation memory
feature_vector_bytes = len(FEATURE_COLUMNS) * np.dtype(np.float32).itemsize
raw_window_bytes = WINDOW_SIZE * np.dtype(np.float32).itemsize

representation_df = pd.DataFrame({
    "representation": ["DSP feature vector", "Raw waveform window"],
    "values_per_window": [len(FEATURE_COLUMNS), WINDOW_SIZE],
    "float32_memory_bytes": [feature_vector_bytes, raw_window_bytes]
})
representation_df["memory_KB"] = (
    representation_df["float32_memory_bytes"] / 1024
)

display(representation_df)
print(
    "Raw-window / feature-vector memory ratio:",
    raw_window_bytes / feature_vector_bytes
)

## 26. Edge-oriented computational profiling

Colab timing is **not an MCU benchmark**. It is a software-level profile of relative computational cost.

Measured quantities:

- DSP + feature extraction latency
- SVM inference latency
- Random Forest inference latency
- 1D-CNN inference latency
- CNN parameter count
- feature-vector memory versus raw-window memory

In [ ]:
# 26. Feature extraction latency
profile_window = raw_X[0]
times = []

for _ in range(10):
    start = time.perf_counter()
    _ = extract_features(butter_lowpass_filter(profile_window))
    times.append(time.perf_counter() - start)

feature_latency = float(np.mean(times))
print(f"Mean DSP + feature extraction latency: {feature_latency:.6f} s/window")

In [ ]:
# 26b. Model inference latency
feature_sample = X_test.iloc[[0]]

svm_times, rf_times = [], []
for _ in range(100):
    start = time.perf_counter()
    _ = tuned_svm.predict(feature_sample)
    svm_times.append(time.perf_counter() - start)

    start = time.perf_counter()
    _ = tuned_rf.predict(feature_sample)
    rf_times.append(time.perf_counter() - start)

svm_latency = float(np.mean(svm_times))
rf_latency = float(np.mean(rf_times))

cnn_sample = X_cnn_test[:1]
_ = cnn_model.predict(cnn_sample, verbose=0)

cnn_times = []
for _ in range(20):
    start = time.perf_counter()
    _ = cnn_model.predict(cnn_sample, verbose=0)
    cnn_times.append(time.perf_counter() - start)

cnn_latency = float(np.mean(cnn_times))

print(f"SVM mean inference latency: {svm_latency:.6f} s/sample")
print(f"RF mean inference latency : {rf_latency:.6f} s/sample")
print(f"CNN mean inference latency: {cnn_latency:.6f} s/sample")

In [ ]:
# 26c. Resource summary
cnn_params = cnn_model.count_params()

edge_summary = pd.DataFrame({
    "item": [
        "DSP + feature extraction",
        "Tuned SVM inference",
        "Tuned Random Forest inference",
        "1D-CNN inference",
        "DSP feature vector memory",
        "Raw waveform window memory",
        "1D-CNN parameters"
    ],
    "value": [
        feature_latency, svm_latency, rf_latency, cnn_latency,
        feature_vector_bytes / 1024, raw_window_bytes / 1024,
        cnn_params
    ],
    "unit": [
        "seconds/window", "seconds/sample", "seconds/sample",
        "seconds/sample", "KB/window", "KB/window", "parameters"
    ]
})

display(edge_summary)

## 27. Reproducibility and validity checks

This section records the exact experiment settings and checks that the evaluation remained recording-aware.

In [ ]:
# 27. Reproducibility report
reproducibility = {
    "seed": SEED,
    "sampling_frequency_hz": FS,
    "lowpass_cutoff_hz": CUTOFF_FREQ,
    "filter_order": FILTER_ORDER,
    "window_seconds": WINDOW_SECONDS,
    "window_samples": WINDOW_SIZE,
    "overlap": OVERLAP,
    "max_windows_per_file": MAX_WINDOWS_PER_FILE,
    "num_recordings_found": int(len(recordings)),
    "num_feature_windows": int(len(features_df)),
    "num_classes": int(NUM_CLASSES),
    "classes": class_names,
    "stratified_group_splits": int(N_SPLITS),
    "holdout_train_recordings": int(groups_train.nunique()),
    "holdout_test_recordings": int(groups_test.nunique()),
    "cnn_train_recordings": int(len(set(raw_groups[cnn_train_idx]))),
    "cnn_validation_recordings": int(len(set(raw_groups[cnn_val_idx]))),
    "cnn_validation_method": best["method"],
    "cnn_validation_class_coverage": float(best["coverage"]),
    "cnn_test_recordings": int(len(set(raw_groups[test_idx_cnn]))),
    "recording_overlap_train_test": len(set(groups_train) & set(groups_test)),
    "recording_overlap_cnn_train_val": len(set(raw_groups[cnn_train_idx]) & set(raw_groups[cnn_val_idx])),
    "recording_overlap_cnn_train_test": len(set(raw_groups[cnn_train_idx]) & set(raw_groups[test_idx_cnn])),
    "recording_overlap_cnn_val_test": len(set(raw_groups[cnn_val_idx]) & set(raw_groups[test_idx_cnn]))
}

print(json.dumps(reproducibility, indent=2))


## 28. Research interpretation guide

The notebook intentionally does **not** declare a single model to be universally "best."

Interpret the results through four questions:

1. **Generalization:** How stable are SVM and Random Forest across grouped folds inside the training recordings?
2. **Representation:** Does adding frequency-domain information change performance relative to time-domain features?
3. **Robustness:** How much does performance change as controlled test noise increases?
4. **Deployment:** How do compact DSP features compare with raw waveform input in memory and software-level computation?

A strong research conclusion should use the complete set of measurements rather than accuracy alone.

### Important dataset limitation

The repository contains relatively few independent recordings in at least one class. Therefore, a perfectly class-complete recording-aware CNN validation set may be impossible. If the notebook reports incomplete validation class coverage, this must be described as a **dataset-structure limitation**, not as evidence of leakage or as a complete multiclass validation result.


## 29. Report-ready conclusion template

After the final run, use the automatically generated tables and the final summary cell below to populate the internship report.

### Dataset and validation
- Number of independent recordings
- Number of segmented windows
- Class names and recording counts
- Recording-level train/test split
- Train/test recording overlap check
- CNN validation method and class coverage

### Classical ML
- Training-only grouped-CV results
- Tuned SVM held-out test metrics
- Tuned Random Forest held-out test metrics

### CNN
- Recording-separated validation performance
- Held-out test metrics
- Parameter count

### DSP representation
- Time-domain vs frequency-domain vs combined ablation
- Random Forest feature-importance analysis

### Robustness
- Clean, 20 dB SNR and 10 dB SNR weighted-F1 measurements
- Performance change under controlled additive noise

### Edge relevance
- DSP feature-vector memory
- Raw waveform memory
- DSP + feature extraction latency
- Model inference latency
- CNN parameter count

### Research claim discipline

The project supports conclusions about the **measured experimental setup and dataset**. It does not establish universal performance across all bearing types, operating conditions, sensors, machines, or embedded processors.

A strong next step is validation on a second dataset and/or hardware benchmarking on a target edge device.


## 30. Optional future extensions

1. Increase the number of windows per recording after confirming runtime.
2. Evaluate different window lengths and filter cutoffs.
3. Add envelope analysis and bearing-characteristic frequency features.
4. Quantize/compress the CNN for TinyML.
5. Benchmark the final pipeline on an actual microcontroller or edge board.
6. Validate the method on a second bearing dataset.
7. Investigate cross-load and cross-speed generalization.

## 30. Automatic final experiment summary and report export

This final section gathers the measured outputs into compact CSV tables and prints a report-ready summary.

No numerical result is manually entered into this cell.


In [ ]:
# 30. Automatic final experiment summary and report export
EXPORT_DIR = Path("/content/oist_final_results")
EXPORT_DIR.mkdir(parents=True, exist_ok=True)

# Export the measured tables.
results_df.to_csv(EXPORT_DIR / "final_heldout_test_metrics.csv", index=False)
cv_results_df.to_csv(EXPORT_DIR / "training_only_grouped_cv.csv", index=False)
cv_summary.to_csv(EXPORT_DIR / "training_only_grouped_cv_summary.csv")
ablation_df.to_csv(EXPORT_DIR / "dsp_ablation.csv", index=False)
robustness_df.to_csv(EXPORT_DIR / "noise_robustness.csv", index=False)
representation_df.to_csv(EXPORT_DIR / "representation_memory.csv", index=False)
edge_summary.to_csv(EXPORT_DIR / "edge_profile.csv", index=False)
importance_df.to_csv(EXPORT_DIR / "random_forest_feature_importance.csv", index=False)
recordings.to_csv(EXPORT_DIR / "recording_audit.csv", index=False)

# Report-ready scalar summary.
summary_rows = []

for row in results_df.to_dict("records"):
    summary_rows.append({
        "section": "held_out_test",
        "item": row["model"],
        "metric": "weighted_f1",
        "value": row["f1_weighted"]
    })

summary_rows.extend([
    {
        "section": "validation",
        "item": "train_test_recording_overlap",
        "metric": "count",
        "value": len(set(groups_train) & set(groups_test))
    },
    {
        "section": "cnn_validation",
        "item": "class_coverage",
        "metric": "fraction",
        "value": best["coverage"]
    },
    {
        "section": "representation",
        "item": "raw_to_feature_memory_ratio",
        "metric": "ratio",
        "value": raw_window_bytes / feature_vector_bytes
    },
    {
        "section": "edge_profile",
        "item": "cnn_parameters",
        "metric": "count",
        "value": cnn_params
    }
])

summary_df = pd.DataFrame(summary_rows)
summary_df.to_csv(EXPORT_DIR / "report_ready_summary.csv", index=False)

print("=" * 72)
print("FINAL OIST EXPERIMENT SUMMARY")
print("=" * 72)
print(f"Independent recordings : {len(recordings)}")
print(f"Feature windows        : {len(features_df)}")
print(f"Classes                : {class_names}")
print(f"Train recordings       : {groups_train.nunique()}")
print(f"Test recordings        : {groups_test.nunique()}")
print(f"Train/test overlap     : {len(set(groups_train) & set(groups_test))}")
print(f"CNN validation method  : {best['method']}")
print(f"CNN validation coverage: {best['coverage']:.1%}")
print()

print("Held-out test weighted F1:")
for row in results_df.itertuples():
    print(f"  {row.model}: {row.f1_weighted:.4f}")

print()
print("Training-only grouped-CV weighted F1:")
for model_name, grp in cv_results_df.groupby("model"):
    print(
        f"  {model_name}: "
        f"{grp['f1_weighted'].mean():.4f} ± {grp['f1_weighted'].std():.4f}"
    )

print()
print(
    f"DSP feature memory : {feature_vector_bytes/1024:.5f} KB/window"
)
print(
    f"Raw waveform memory: {raw_window_bytes/1024:.3f} KB/window"
)
print(
    f"Memory ratio       : {raw_window_bytes/feature_vector_bytes:.1f}x"
)

if best["coverage"] < 1.0:
    print(
        "\nCNN validation note: validation does not contain every class. "
        "Report this explicitly as a limitation caused by the available "
        "independent recordings."
    )

print(
    "\nEdge-profile note: timing values were measured in Colab and are "
    "not MCU hardware benchmarks."
)

print(f"\nCSV results exported to: {EXPORT_DIR}")
print("Final leakage check:", "PASS" if len(set(groups_train) & set(groups_test)) == 0 else "FAIL")


## 31. Final limitations and next-step experiments

### Current limitations
- The experiment is based on the CWRU repository used by the original project.
- The number of independent recordings is much smaller than the number of segmented windows; therefore, window count must not be mistaken for independent sample count.
- Some classes, particularly the smallest class in this repository, constrain class-complete recording-aware validation.
- Software timings from Colab cannot substitute for an embedded hardware benchmark.
- Controlled additive Gaussian noise is only one robustness scenario.

### Strong next experiments
1. Validate the complete pipeline on a second bearing dataset.
2. Test cross-load and cross-speed generalization.
3. Evaluate additional window lengths and filter cutoffs.
4. Add envelope analysis and bearing-characteristic frequency features.
5. Benchmark DSP + classical ML on an actual MCU/edge processor.
6. Quantize or compress the CNN if raw-signal inference is pursued for TinyML deployment.

These extensions are deliberately kept separate from the final held-out evaluation so that the current results remain auditable.
